# Subgen — Subdomain Generation Model

Full pipeline: preprocess → train tokenizer → pack dataset → train → generate

**Requirements:** Upload `domains_export.csv` to Google Drive or directly to Colab.

## 0. Setup

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Set work directory — change this if you want a different location
WORK_DIR = '/content/drive/MyDrive/subgen'

import os
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install dependencies
!pip install -q torch transformers tokenizers tldextract wandb numpy tqdm accelerate

In [ ]:
# Check GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
    print(f'BF16 support: {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No GPU detected. Training will be very slow.')

In [ ]:
# Clone or update repo
REPO_DIR = os.path.join(WORK_DIR, 'repo')

if os.path.isdir(REPO_DIR):
    print('Repo already exists, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/c3l3si4n/subgen.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Repo dir: {os.getcwd()}')
!ls

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

## 1. Preprocess

In [ ]:
# Verify dataset exists
DATASET_PATH = os.path.join(REPO_DIR, 'dataset', 'domains_export.csv')
if not os.path.isfile(DATASET_PATH):
    # Try uploading
    from google.colab import files
    print('Dataset not found. Upload domains_export.csv:')
    uploaded = files.upload()
    os.makedirs(os.path.join(REPO_DIR, 'dataset'), exist_ok=True)
    for name, data in uploaded.items():
        with open(DATASET_PATH, 'wb') as f:
            f.write(data)
        break

!wc -l {DATASET_PATH} | head -1

In [ ]:
from data.preprocess import preprocess

preprocess(
    input_path=DATASET_PATH,
    output_dir=os.path.join(REPO_DIR, 'data'),
    max_subs_per_domain=500,
)

In [ ]:
# Verify: check a few lines
!head -3 data/train_sequences.txt

## 2. Train Tokenizer

In [ ]:
from data.tokenizer_train import train_tokenizer

tokenizer = train_tokenizer(
    input_files=[os.path.join(REPO_DIR, 'data', 'train_sequences.txt')],
    output_dir=os.path.join(REPO_DIR, 'tokenizer'),
    vocab_size=8192,
)

## 3. Pack Dataset

In [ ]:
from data.dataset import pretokenize_packed
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained(os.path.join(REPO_DIR, 'tokenizer'))

for split in ['train', 'val']:
    input_path = os.path.join(REPO_DIR, 'data', f'{split}_sequences.txt')
    output_path = os.path.join(REPO_DIR, 'data', f'{split}.bin')
    if os.path.exists(input_path):
        pretokenize_packed(input_path, output_path, tokenizer)
    else:
        print(f'Skipping {split}: not found')

## 4. Train

In [ ]:
# Optional: set up W&B logging
USE_WANDB = False  # Set True and run `wandb login` if you want logging

if USE_WANDB:
    import wandb
    wandb.login()

In [ ]:
from data.dataset import DomainDataset
from model.config import build_config, build_model
from transformers import PreTrainedTokenizerFast, Trainer, TrainingArguments

tokenizer = PreTrainedTokenizerFast.from_pretrained(os.path.join(REPO_DIR, 'tokenizer'))

config = build_config(vocab_size=len(tokenizer))
model = build_model(config)

train_dataset = DomainDataset(os.path.join(REPO_DIR, 'data', 'train.bin'))
val_dataset = DomainDataset(os.path.join(REPO_DIR, 'data', 'val.bin'))
print(f'Train rows: {len(train_dataset)}')
print(f'Val rows: {len(val_dataset)}')

In [ ]:
# Detect hardware capabilities
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

# T4 has 16GB VRAM — use smaller batch size than local 9070 XT
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0
batch_size = 32 if vram_gb >= 20 else 16 if vram_gb >= 14 else 8
grad_accum = max(1, 128 // batch_size)  # effective batch ~128

print(f'VRAM: {vram_gb:.1f} GB → batch_size={batch_size}, grad_accum={grad_accum}')
print(f'Precision: {"bf16" if use_bf16 else "fp16" if use_fp16 else "fp32"}')

In [ ]:
OUTPUT_DIR = os.path.join(REPO_DIR, 'checkpoints')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=6e-4,
    warmup_steps=2000,
    weight_decay=0.1,
    max_grad_norm=1.0,
    adam_beta1=0.9,
    adam_beta2=0.95,
    lr_scheduler_type='cosine',
    bf16=use_bf16,
    fp16=use_fp16,
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=1000,
    save_steps=2000,
    save_total_limit=2,
    report_to='wandb' if USE_WANDB else 'none',
    run_name='subgen-150m-colab',
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    torch_compile=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
# Auto-resume from checkpoint if available
resume_from = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')]
    if checkpoints:
        latest = max(checkpoints, key=lambda x: int(x.split('-')[1]))
        resume_from = os.path.join(OUTPUT_DIR, latest)
        print(f'Resuming from {resume_from}')

trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# Save final model
final_dir = os.path.join(OUTPUT_DIR, 'final')
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'Final model saved to {final_dir}')

## 5. Generate

In [ ]:
from generate import generate_subdomains, load_model

model, tokenizer = load_model(final_dir, device='auto')

candidates = generate_subdomains(
    model, tokenizer,
    root_domain='example.com',
    num_samples=20,
    temperature=0.8,
)

print(f'Generated {len(candidates)} candidates:')
for c in candidates:
    print(c)